# 02. Processamento de Microdados: Firm-Level (SABI)
**Projeto:** Modelação de episódios recorrentes de stress financeiro em PME

Este notebook detalha o processo de limpeza e transformação dos dados financeiros, de *governance* e indicadores de eficiência das PME. O objetivo central é a transição de um formato de dados estático (**Wide Format**) para uma estrutura de painel longitudinal (**Long Format**). Esta transformação é mandatória para que os modelos de análise de sobrevivência capturem covariáveis dependentes do tempo e a **memória temporal** das empresas ao longo do ciclo saúde-doença-recaída.

In [ ]:
import pandas as pd
import os
import numpy as np
import re

MICRO_RAW_PATH = "../data/raw/micro"
INTERIM_PATH = "../data/interim"
os.makedirs(INTERIM_PATH, exist_ok=True)

tec_files = sorted([f for f in os.listdir(MICRO_RAW_PATH) if f.startswith("sabi_tec_") and "gov" not in f and "extra" not in f])
gov_files = sorted([f for f in os.listdir(MICRO_RAW_PATH) if f.startswith("sabi_tec_gov_")])
extra_files = sorted([f for f in os.listdir(MICRO_RAW_PATH) if "extra" in f])

## 1. Consolidação e Integração Multidimensional
Fundimos as exportações fragmentadas do SABI (Financeiros, Governance e Indicadores Extra). O merge é realizado utilizando o **NIF Code** como chave única. Esta etapa consolida a visão 360º de cada empresa, unindo o desempenho financeiro à estrutura de gestão e eficiência operacional.

In [8]:
all_main_chunks = []
for f_tec, f_gov in zip(tec_files, gov_files):
    df_tec = pd.read_csv(os.path.join(MICRO_RAW_PATH, f_tec), encoding='utf-16', sep='\t', low_memory=False)
    df_gov = pd.read_csv(os.path.join(MICRO_RAW_PATH, f_gov), encoding='utf-16', sep='\t', low_memory=False)
    df_gov = df_gov.drop(columns=[c for c in ['Mark', 'Company Name'] if c in df_gov.columns])
    
    df_chunk = pd.merge(df_tec, df_gov, on='NIF Code', how='outer', suffixes=('', '_gov'))
    all_main_chunks.append(df_chunk)

df_micro = pd.concat(all_main_chunks, ignore_index=True)

all_extra = []
for f_extra in extra_files:
    df_ex = pd.read_csv(os.path.join(MICRO_RAW_PATH, f_extra), encoding='utf-16', sep='\t', low_memory=False)
    all_extra.append(df_ex)

df_extra_full = pd.concat(all_extra, ignore_index=True)
df_extra_full = df_extra_full.drop(columns=[c for c in ['Mark', 'Company Name'] if c in df_extra_full.columns])

df_micro = pd.merge(df_micro, df_extra_full, on='NIF Code', how='left', suffixes=('', '_extra'))

cols_to_drop = ['Mark', 'Company Name_gov', 'Company Name.1', 'Company Name_extra']
df_micro = df_micro.drop(columns=[c for c in cols_to_drop if c in df_micro.columns])

## 2. Padronização de Atributos e Nomenclatura
Normalizamos os cabeçalhos das novas variáveis (PMR, PMP, Liquidez Estrita, Fundo de Maneio) para o formato `Variavel_Ano`. Isto permite uma fusão coerente de todas as dimensões de dados no formato longitudinal.

In [9]:
def get_standardized_mapping(columns):
    patterns = {
        r'Operating revenue.* EUR (\d{4})': r'Revenue_\1',
        r'Number of employees (\d{4})': r'Employees_\1',
        r'Total assets.* EUR (\d{4})': r'TotalAssets_\1',
        r"Shareholders' equity.* EUR (\d{4})": r'Equity_\1',
        r'Economic profitability.* (\d{4})': r'ROA_\1',
        r'Financial profitability.* (\d{4})': r'ROE_\1',
        r'General liquidity.* (\d{4})': r'Liquidity_\1',
        r'Indebtness.* (\d{4})': r'Indebtedness_\1',
        r'P/L for period.* EUR (\d{4})': r'NetProfit_\1',
        r'EBITDA.* EUR (\d{4})': r'EBITDA_\1',
        r'Interests and similary expenses.* EUR (\d{4})': r'Interests_\1',
        r'Retained earnings.* EUR (\d{4})': r'RetainedEarnings_\1',
        r'Collection period \(days\) days (\d{4})': r'PMR_\1',
        r'Credit Period \(days\) days (\d{4})': r'PMP_\1',
        r'Liquidity Ratio % (\d{4})': r'AcidTest_\1',
        r'Cash flow th EUR (\d{4})': r'CashFlow_\1'
    }
    
    mapping = {
        'Number of current directors & managers': 'BoardSize',
        'BvD Independence Indicator': 'IndependenceIndicator',
        'Shareholder - Total %': 'OwnershipConcentration',
        'CAE Rev.3 Primary Code': 'CAE',
        'District': 'District',
        'Legal Form': 'LegalForm'
    }
    
    for col in columns:
        if col in mapping: continue
        new_name = col
        for pattern, replacement in patterns.items():
            if re.search(pattern, col):
                new_name = re.sub(pattern, replacement, col)
                break
        mapping[col] = new_name
    return mapping

df_micro = df_micro.rename(columns=get_standardized_mapping(df_micro.columns))

## 3. Transformação para Formato Longo (Painel)


In [10]:
id_vars = [
    'NIF Code', 'Company Name', 'Date of Establishment', 'Status', 'Status Date', 
    'BoardSize', 'IndependenceIndicator', 'OwnershipConcentration',
    'CAE', 'District', 'LegalForm'
]
stubnames = [
    'Revenue', 'Employees', 'TotalAssets', 'Equity', 'ROA', 'ROE', 
    'Liquidity', 'Indebtedness', 'NetProfit', 'EBITDA', 'Interests', 'RetainedEarnings',
    'PMR', 'PMP', 'AcidTest', 'CashFlow'
]

df_long = pd.wide_to_long(
    df_micro, 
    stubnames=stubnames, 
    i=id_vars, 
    j='Year', 
    sep='_', 
    suffix='\d+'
).reset_index()

## 4. Limpeza de Atributos e Engenharia de Features
Procedemos à conversão numérica rigorosa e tratamos o CAE para extrair o setor de atividade (2 dígitos). Calculamos também a idade da empresa, variável estática fundamental para modelos de sobrevivência.

In [11]:
numeric_cols = stubnames + ['BoardSize', 'OwnershipConcentration']
for col in numeric_cols:
    if col in df_long.columns:
        df_long[col] = pd.to_numeric(df_long[col].astype(str).str.replace(',', '.'), errors='coerce')

df_long['CAE_Sector'] = df_long['CAE'].astype(str).str[:2]

df_long['Year'] = df_long['Year'].astype(int)
df_long['Date of Establishment'] = pd.to_datetime(df_long['Date of Establishment'], dayfirst=True, errors='coerce')
df_long['FirmAge'] = df_long['Year'] - df_long['Date of Establishment'].dt.year

df_long = df_long.dropna(subset=['Revenue', 'TotalAssets'], how='all')

print(f"Dataset consolidado: {len(df_long)} observações Empresa-Ano.")

Dataset consolidado: 127621 observações Empresa-Ano.


## 5. Conclusão da Fase de Microdados
A base de dados microeconómica está agora enriquecida com dimensões de **Eficiência Operacional** e **Contexto Setorial/Geográfico**. Esta profundidade de dados permitirá aos modelos Deep Learning detetar sinais precoces de stress que rácios financeiros isolados poderiam omitir.

In [12]:
output_file = os.path.join(INTERIM_PATH, "micro_long.csv")
df_long.to_csv(output_file, index=False)
print(f"Ficheiro final exportado: {output_file}")

Ficheiro final exportado: ../data/interim\micro_long.csv


## 6. Dicionário de Variáveis (Versão Final Consolidada)

Este codebook detalha a totalidade dos atributos (financeiros, estruturais e de gestão) que compõem o dataset longitudinal para a modelação da sobrevivência das PME.

### 6.1 Variáveis de Identificação e Contexto (Static Features)
| Variável | Definição Técnica | Tipo | Fonte |
| :--- | :--- | :--- | :--- |
| `NIF Code` | Número de Identificação Fiscal (Chave Única da Empresa). | ID | SABI |
| `Company Name` | Nome oficial da entidade jurídica. | Texto | SABI |
| `Year` | Ano civil da observação financeira. | Temporal | SABI |
| `FirmAge` | Idade da empresa em anos desde a constituição oficial. | Numérica | SABI |
| `CAE_Sector (Economic Activity Code)` | Divisão CAE Rev. 3 (primeiros 2 dígitos - Setor de Atividade). | Categórica | SABI |
| `District` | Localização da sede da empresa (Distrito/Região). | Categórica | SABI |
| `LegalForm` | Estrutura jurídica (Lda, SA, Unipessoal, etc.). | Categórica | SABI |
| `Status` | Estado jurídico atual no SABI (Ativa, Insolvente, Liquidação, etc.). | Categórica | SABI |

### 6.2 Performance Financeira (Time-Varying Covariates)
| Variável | Descrição Técnica (SABI) | Unidade | Fonte |
| :--- | :--- | :--- | :--- |
| `Revenue` | Volume de Negócios / Proveitos Operacionais. | milhares EUR | SABI |
| `Employees` | Número médio de empregados durante o exercício. | Unidades | SABI |
| `TotalAssets` | Ativo Total (Soma do Ativo Corrente e Não Corrente). | milhares EUR | SABI |
| `Equity` | Capitais Próprios (Capital Social + Reservas + Resultados). | milhares EUR | SABI |
| `NetProfit (Liquid Result)` | Lucro ou Prejuízo do exercício após impostos. | milhares EUR | SABI |
| `EBITDA (Earnings Before Interest, Taxes, Depreciation and Amortization)` | Resultados antes de juros, impostos, amortizações e provisões. | milhares EUR | SABI |
| `Interests` | Gastos de Financiamento (Juros e gastos similares). | milhares EUR | SABI |
| `RetainedEarnings (Transitioned results)` | Lucros acumulados não distribuídos de exercícios anteriores. | milhares EUR | SABI |
| `Liquidity` | Liquidez Geral (Rácio entre Ativo Corrente e Passivo Corrente). | Rácio | SABI |
| `Indebtedness` | Rácio entre Passivo Total e Capitais Próprios. | % | SABI |
| `ROA (Return on Assets)` | Rentabilidade Económica (Lucro Operacional / Ativo Total). | % | SABI |
| `ROE (Return on Equity)` | Rentabilidade Financeira (Lucro Líquido / Capitais Próprios). | % | SABI |

### 6.3 Eficiência Operacional e Solvabilidade (Extra Features)
| Variável | Descrição Técnica (SABI) | Unidade | Fonte |
| :--- | :--- | :--- | :--- |
| `PMR (Periodo Médio de Recebimento)` | Período Médio de Recebimento de clientes. | Dias | SABI |
| `PMP (Periodo Médio de Pagamento)` | Período Médio de Pagamento a fornecedores. | Dias | SABI |
| `AcidTest (Reduced Liquidity)` | Rácio [(Ativo Corrente - Inventários) / Passivo Corrente]. | Rácio | SABI |
| `CashFlow`| Fluxo de caixa gerado pela empresa. | milhares EUR | SABI |

### 6.4 Corporate Governance (Non-Financial Features)
| Variável | Significado Técnico | Fonte |
| :--- | :--- | :--- |
| `BoardSize` | Número total de diretores e gestores em funções. | SABI |
| `IndependenceIndicator`| Grau de autonomia da gestão face aos acionistas. | SABI |
| `OwnershipConcentration`| Percentagem de capital detido pelo maior acionista. | SABI |